# Análise Exploratória da Inadimplência em Clientes de Cartão de Crédito

## Objetivo

Este projeto busca responder à seguinte pergunta:

> **Quais características e comportamentos financeiros estão mais associados à inadimplência dos clientes?**

A coluna `default` será utilizada como variável resposta:

- `0`: cliente adimplente;
- `1`: cliente inadimplente.

O projeto contém:

1. exploração dos dados;
2. transformação e limpeza;
3. visualização;
4. geração de insights;
5. conclusão e recomendações.

> **Importante:** esta é uma análise exploratória. As relações encontradas representam associações e não comprovam causalidade.


## 1. Importação das bibliotecas

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)


## 2. Leitura do conjunto de dados

O código abaixo procura automaticamente um arquivo CSV dentro do ambiente do Kaggle.  
Também existe uma alternativa para executar o notebook localmente.


In [ ]:
# Procura arquivos CSV no Kaggle
arquivos_kaggle = list(Path("/kaggle/input").rglob("*.csv"))

if arquivos_kaggle:
    caminho_arquivo = arquivos_kaggle[0]
else:
    # Alternativa para execução local
    caminho_arquivo = Path("Python_M11_support material.csv")

print(f"Arquivo utilizado: {caminho_arquivo}")

# O texto "na" será interpretado como valor ausente
df = pd.read_csv(caminho_arquivo, na_values="na")

df.head()


## 3. Exploração inicial

In [ ]:
print(f"Quantidade de linhas: {df.shape[0]}")
print(f"Quantidade de colunas: {df.shape[1]}")
print(f"Linhas duplicadas: {df.duplicated().sum()}")

df.info()


In [ ]:
# Amostra aleatória dos dados
df.sample(5, random_state=42)


### 3.1 Distribuição da variável `default`

In [ ]:
distribuicao_default = (
    df["default"]
    .value_counts()
    .sort_index()
    .rename(index={0: "Adimplente", 1: "Inadimplente"})
)

percentuais_default = (
    df["default"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .rename(index={0: "Adimplente", 1: "Inadimplente"})
)

resumo_default = pd.DataFrame({
    "quantidade": distribuicao_default,
    "percentual": percentuais_default.round(2)
})

resumo_default


In [ ]:
ax = resumo_default["quantidade"].plot(
    kind="bar",
    figsize=(8, 5),
    title="Distribuição dos clientes por situação de pagamento"
)

ax.set_xlabel("Situação")
ax.set_ylabel("Quantidade de clientes")
ax.tick_params(axis="x", rotation=0)

for indice, valor in enumerate(resumo_default["quantidade"]):
    ax.text(indice, valor, f"{valor:,}".replace(",", "."), ha="center", va="bottom")

plt.tight_layout()
plt.show()


**Leitura inicial:** a base possui mais clientes adimplentes do que inadimplentes.  
Esse desequilíbrio deve ser considerado ao interpretar percentuais e comparações.


### 3.2 Tipos das colunas

In [ ]:
df.dtypes.to_frame(name="tipo")


As colunas `limite_credito` e `valor_transacoes_12m` representam valores monetários, mas foram lidas como texto.  
Isso acontece porque os valores utilizam ponto como separador de milhar e vírgula como separador decimal.


### 3.3 Dados ausentes

In [ ]:
dados_ausentes = pd.DataFrame({
    "quantidade": df.isna().sum(),
    "percentual": (df.isna().mean() * 100).round(2)
})

dados_ausentes = dados_ausentes[dados_ausentes["quantidade"] > 0]
dados_ausentes


In [ ]:
if not dados_ausentes.empty:
    ax = dados_ausentes["percentual"].sort_values().plot(
        kind="barh",
        figsize=(8, 4),
        title="Percentual de dados ausentes por coluna"
    )
    ax.set_xlabel("Percentual (%)")
    ax.set_ylabel("Coluna")
    plt.tight_layout()
    plt.show()


## 4. Transformação e limpeza dos dados

### 4.1 Conversão das colunas monetárias

In [ ]:
def converter_valor_monetario(valor):
    """Converte texto no formato brasileiro para número decimal."""
    if pd.isna(valor):
        return np.nan

    return float(
        str(valor)
        .replace(".", "")
        .replace(",", ".")
    )


colunas_monetarias = ["limite_credito", "valor_transacoes_12m"]

for coluna in colunas_monetarias:
    df[coluna] = df[coluna].apply(converter_valor_monetario)

df[colunas_monetarias].head()


In [ ]:
df[colunas_monetarias].dtypes


### 4.2 Tratamento dos dados ausentes

Neste projeto, as linhas com valores ausentes serão removidas para manter o processo simples e alinhado ao objetivo da atividade.

Como essa decisão elimina uma parcela relevante da base, será verificado se a proporção de inadimplentes mudou muito após a limpeza.


In [ ]:
linhas_antes = len(df)
taxa_antes = df["default"].mean() * 100

df_limpo = df.dropna().copy()

linhas_depois = len(df_limpo)
taxa_depois = df_limpo["default"].mean() * 100

print(f"Linhas antes da limpeza: {linhas_antes}")
print(f"Linhas depois da limpeza: {linhas_depois}")
print(f"Linhas removidas: {linhas_antes - linhas_depois}")
print(f"Percentual removido: {(linhas_antes - linhas_depois) / linhas_antes * 100:.2f}%")
print()
print(f"Taxa de inadimplência antes: {taxa_antes:.2f}%")
print(f"Taxa de inadimplência depois: {taxa_depois:.2f}%")


In [ ]:
print(f"Valores ausentes restantes: {df_limpo.isna().sum().sum()}")
print(f"Linhas duplicadas restantes: {df_limpo.duplicated().sum()}")


## 5. Análise exploratória e visualização

### 5.1 Comparação das médias entre adimplentes e inadimplentes

In [ ]:
colunas_numericas = [
    "idade",
    "dependentes",
    "meses_de_relacionamento",
    "qtd_produtos",
    "iteracoes_12m",
    "meses_inativo_12m",
    "limite_credito",
    "valor_transacoes_12m",
    "qtd_transacoes_12m"
]

comparacao_medias = (
    df_limpo
    .groupby("default")[colunas_numericas]
    .mean()
    .transpose()
    .rename(columns={0: "Adimplentes", 1: "Inadimplentes"})
)

comparacao_medias["diferença_percentual"] = (
    (comparacao_medias["Inadimplentes"] - comparacao_medias["Adimplentes"])
    / comparacao_medias["Adimplentes"]
    * 100
)

comparacao_medias.round(2)


### 5.2 Quantidade de transações nos últimos 12 meses

In [ ]:
dados_boxplot = [
    df_limpo.loc[df_limpo["default"] == 0, "qtd_transacoes_12m"],
    df_limpo.loc[df_limpo["default"] == 1, "qtd_transacoes_12m"]
]

plt.figure(figsize=(8, 5))
plt.boxplot(dados_boxplot, labels=["Adimplentes", "Inadimplentes"])
plt.title("Quantidade de transações por situação do cliente")
plt.ylabel("Quantidade de transações em 12 meses")
plt.tight_layout()
plt.show()


**Insight:** clientes inadimplentes apresentam, em média, uma quantidade menor de transações.  
Isso indica que a redução da movimentação do cartão pode ser um sinal relevante de risco.


### 5.3 Valor total transacionado nos últimos 12 meses

In [ ]:
dados_boxplot = [
    df_limpo.loc[df_limpo["default"] == 0, "valor_transacoes_12m"],
    df_limpo.loc[df_limpo["default"] == 1, "valor_transacoes_12m"]
]

plt.figure(figsize=(8, 5))
plt.boxplot(dados_boxplot, labels=["Adimplentes", "Inadimplentes"], showfliers=False)
plt.title("Valor transacionado por situação do cliente")
plt.ylabel("Valor das transações em 12 meses")
plt.tight_layout()
plt.show()


**Insight:** o grupo inadimplente movimenta menos dinheiro no cartão.  
A combinação entre poucas transações e baixo valor movimentado reforça a hipótese de menor engajamento financeiro.


### 5.4 Quantidade de produtos contratados

In [ ]:
taxa_por_produtos = (
    df_limpo
    .groupby("qtd_produtos")["default"]
    .agg(quantidade="size", taxa_inadimplencia="mean")
)

taxa_por_produtos["taxa_inadimplencia"] *= 100
taxa_por_produtos.round(2)


In [ ]:
ax = taxa_por_produtos["taxa_inadimplencia"].plot(
    kind="bar",
    figsize=(9, 5),
    title="Taxa de inadimplência por quantidade de produtos contratados"
)

ax.set_xlabel("Quantidade de produtos")
ax.set_ylabel("Taxa de inadimplência (%)")
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()


**Insight:** clientes com poucos produtos contratados apresentam uma taxa maior de inadimplência.  
Uma relação mais ampla com a instituição pode estar associada a maior permanência e regularidade.


### 5.5 Interações com o cliente nos últimos 12 meses

In [ ]:
taxa_por_interacoes = (
    df_limpo
    .groupby("iteracoes_12m")["default"]
    .agg(quantidade="size", taxa_inadimplencia="mean")
)

taxa_por_interacoes["taxa_inadimplencia"] *= 100
taxa_por_interacoes.round(2)


In [ ]:
ax = taxa_por_interacoes["taxa_inadimplencia"].plot(
    kind="bar",
    figsize=(9, 5),
    title="Taxa de inadimplência por quantidade de interações"
)

ax.set_xlabel("Interações nos últimos 12 meses")
ax.set_ylabel("Taxa de inadimplência (%)")
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()


**Insight:** a taxa de inadimplência aumenta entre clientes com mais interações registradas.  
Uma interpretação possível é que parte dessas interações esteja ligada a problemas, cobranças ou tentativas de atendimento.  
O conjunto de dados, porém, não informa o motivo das interações.


### 5.6 Meses de inatividade

In [ ]:
taxa_por_inatividade = (
    df_limpo
    .groupby("meses_inativo_12m")["default"]
    .agg(quantidade="size", taxa_inadimplencia="mean")
)

taxa_por_inatividade["taxa_inadimplencia"] *= 100
taxa_por_inatividade.round(2)


In [ ]:
ax = taxa_por_inatividade["taxa_inadimplencia"].plot(
    kind="bar",
    figsize=(9, 5),
    title="Taxa de inadimplência por meses de inatividade"
)

ax.set_xlabel("Meses inativo nos últimos 12 meses")
ax.set_ylabel("Taxa de inadimplência (%)")
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()


**Insight:** em boa parte da distribuição, mais meses de inatividade estão associados a uma taxa maior de inadimplência.  
Algumas categorias possuem poucas observações e devem ser interpretadas com cautela.


### 5.7 Variáveis categóricas

In [ ]:
def plotar_taxa_categoria(dataframe, coluna, titulo):
    tabela = (
        dataframe
        .groupby(coluna)["default"]
        .agg(quantidade="size", taxa_inadimplencia="mean")
        .sort_values("taxa_inadimplencia", ascending=False)
    )

    tabela["taxa_inadimplencia"] *= 100

    ax = tabela["taxa_inadimplencia"].plot(
        kind="bar",
        figsize=(10, 5),
        title=titulo
    )

    ax.set_xlabel(coluna.replace("_", " ").title())
    ax.set_ylabel("Taxa de inadimplência (%)")
    ax.tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()

    return tabela.round(2)


In [ ]:
plotar_taxa_categoria(
    dataframe=df_limpo,
    coluna="sexo",
    titulo="Taxa de inadimplência por sexo"
)


In [ ]:
plotar_taxa_categoria(
    dataframe=df_limpo,
    coluna="escolaridade",
    titulo="Taxa de inadimplência por escolaridade"
)


In [ ]:
plotar_taxa_categoria(
    dataframe=df_limpo,
    coluna="salario_anual",
    titulo="Taxa de inadimplência por faixa de salário anual"
)


As diferenças entre algumas categorias são pequenas ou possuem amostras desiguais.  
Por isso, características demográficas isoladas parecem menos úteis do que indicadores de comportamento financeiro.


### 5.8 Matriz de correlação

In [ ]:
correlacao = (
    df_limpo
    .drop(columns="id")
    .select_dtypes(include="number")
    .corr()
)

fig, ax = plt.subplots(figsize=(11, 8))
imagem = ax.imshow(correlacao, aspect="auto")

ax.set_xticks(range(len(correlacao.columns)))
ax.set_xticklabels(correlacao.columns, rotation=90)

ax.set_yticks(range(len(correlacao.index)))
ax.set_yticklabels(correlacao.index)

fig.colorbar(imagem, ax=ax, label="Correlação")
ax.set_title("Matriz de correlação das variáveis numéricas")

plt.tight_layout()
plt.show()


## 6. Storytelling: o que os dados contam?

A análise mostra que a inadimplência está mais associada ao **comportamento de uso do cartão** do que às características demográficas.

Os clientes inadimplentes apresentam, em média:

- menos produtos contratados;
- menor quantidade de transações;
- menor valor total movimentado;
- mais meses de inatividade;
- maior quantidade de interações registradas com a instituição.

Esses resultados sugerem um possível processo de afastamento: o cliente passa a utilizar menos o cartão, movimenta menos dinheiro, permanece inativo por mais tempo e registra mais interações com a instituição.

Características como idade, tempo de relacionamento e limite de crédito apresentam diferenças menores entre os grupos e, isoladamente, parecem menos decisivas.


## 7. Conclusão

Os indicadores mais relevantes encontrados nesta análise foram:

1. `qtd_transacoes_12m`;
2. `valor_transacoes_12m`;
3. `qtd_produtos`;
4. `meses_inativo_12m`;
5. `iteracoes_12m`.

Uma instituição financeira poderia acompanhar clientes que apresentam queda na movimentação, aumento da inatividade e crescimento no número de interações.

Possíveis ações:

- criar alertas para redução brusca de transações;
- oferecer contato preventivo e renegociação;
- investigar o motivo das interações mais frequentes;
- desenvolver campanhas de reativação;
- avaliar clientes com poucos produtos e baixa utilização.

### Limitações

- Foram removidas linhas com dados ausentes;
- o conjunto não informa o motivo das interações;
- associação estatística não significa causalidade;
- não foi construído um modelo preditivo, pois o foco deste projeto é exploração e visualização.


## 8. Checklist antes da publicação no Kaggle

- [ ] Executar todas as células sem erro;
- [ ] conferir se todos os gráficos aparecem;
- [ ] revisar título, textos e conclusões;
- [ ] acrescentar seu nome na introdução;
- [ ] salvar uma versão completa;
- [ ] deixar o notebook público;
- [ ] copiar o link e enviar para a tutoria.
